In [1]:
# Base class
from src.Base.Config import Config

# Project class
from src.BetaMouv.Trajectory import TrajectoryDLC, DLCTracking
import src.BetaMouv.Behavior as Behavior
import src.utils as u

# other lib
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### Load project from config

In [2]:
cfg = Config(Path("./config.yaml"))
project = cfg.load_project()

log: config_20261023_111926.yaml

 CONTRA_CHR_#517_#531


Loading joblib: 100%|██████████| 58/58 [00:25<00:00,  2.32it/s]

--> 17/58 FILES LOADED
--> 1541 trials


### Setup analysis dir

In [ ]:
custom_params = {"axes.spines.right": False, "axes.spines.top": False}
sns.set_theme("talk", style="ticks", rc=custom_params, palette="pastel")

analysis_dir = project.paths.analysis(project.trialgroup.keep_val)
df_dir = u.make_dir(analysis_dir / "trials_df")
u.make_dir(analysis_dir)


PosixPath('data/BetaMouv/results/CONTRA_CHR_#517_#531')

Some hypper parameters

In [4]:
project.trialgroup.crop_laser_period = True
project.trialgroup.remove_NOstim = True

In [9]:
traj_data = []

n=0
for trial in project.trialgroup.trials:
    if not trial.is_valid():
        continue

    for bp, stat in trial.trajectories.items():
        data = {info: val for  info, val in stat.items()}
        data["id"] = n
        data["trial_name"] = trial.name

        traj_data.append(data)


traj_data_df = pd.DataFrame(traj_data)
traj_data_df.to_csv("traj_data_df.csv")

In [14]:
project.trialgroup.build_df(df_dir, "trial_success")

AttributeError: 'TrialGroup' object has no attribute 'build_df'

In [ ]:
project.trialgroup.trial_success_rate(u.make_path(analysis_dir, "trial_success_rate.png"))
project.trialgroup.trajectories_success_rate(u.make_path(analysis_dir, "trajectories_success_rate.png"))

project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"),
                                  subfig_group="subject", 
                                  groups=["laser_type"],
                                  title=project.trialgroup.group_name)

project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"), 
                                  subfig_group="condition", 
                                  groups=["subject", "laser_type", "laser_intensity"],
                                  title=project.trialgroup.group_name)

project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"), 
                                  subfig_group=None, 
                                  groups=["subject", "is_valid", "failing_step"],
                                  title=project.trialgroup.group_name)



## Behavior analysis

In [ ]:
features = [
            "relative_t",
            "x", 
            "y",
            "area", 
            "centroid_direction", 
            "centroid_velocity", 
            "centroid_speed", 
            "centroid_acc", 
            "f3_sp_distance", 
            "f3_sp_angle", 
            "centroid_lever_dist"
]

In [ ]:
for feat in features: 
    project.trialgroup.plot_tendency_beha_features(
                        save_as=u.make_path(analysis_dir / "behavior" / "with_behavior_span", f"{feat}.png"),
                        x="relative_t", y=feat, show_beha="spatial_behavior")

    project.trialgroup.plot_tendency_beha_features(
                        save_as=u.make_path(analysis_dir / "behavior", f"{feat}.png"),
                        x="relative_t", y=feat, show_beha=None)

project.trialgroup.lineplot_traj_centroid(save_as=u.make_path(analysis_dir / "behavior", f"centroid_traj.png"),) 


Distribution of each features, per protocols

In [ ]:
for l_type in ["Beta","Conti"] : 
    for l_intensity in ["low", "high"]: 
        for l_state in ["LaserOn", "LaserOff"] : 
            project.trialgroup.distri_feat_per_behavior(laser_type=l_type, laser_intensity=l_intensity, laser_state=l_state,
                                                    save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", f"spatial_behavior_{l_type}_{l_intensity}_{l_state}.svg"),
                                                    features_list=features)

            project.trialgroup.distri_feat_per_behavior(data=df, behavior_varname="pred_state", laser_type=l_type, laser_intensity=l_intensity, laser_state=l_state,
                                                    save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", f"HMM_pred_state_{l_type}_{l_intensity}_{l_state}.svg"),
                                                    features_list=features)



Distribution of each features, for the whole dataset

In [ ]:
project.trialgroup.distri_feat_per_behavior(
                                        save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", "spatial_behavior_state_all.svg"),
                                        features_list=features)

project.trialgroup.distri_feat_per_behavior(data=df, behavior_varname="pred_state", 
                                        save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", "HMM_pred_state_all.svg"),
                                        features_list=features)



Behavior exploration : HMM fitting

In [ ]:
df = project.trialgroup.behavior_features_df(add_identity=True)
Behavior.HMM_exploration(df, analysis_dir, init=False)


## Trajectory analysis

In [ ]:
timeseries_metric = [
                    "x", 
                    "y", 
                    "instant_velocity", 
                    "instant_speed", 
                    "acc", 
                    "signed_acc", 
                    "lever_distance",
                    "angle"
                    ]

bodypart = "finger_3" 

In [ ]:
for val in timeseries_metric:
    project.trialgroup.plot_tendency_metrics(
        value=val,
        save_as=u.make_path(analysis_dir / "tendency" / bodypart, f"{val}.svg"),
        # show_units=True, 
    )

Lineplot traj with behavior boxes

In [ ]:
trial = project.trialgroup.trials[1]
behavior_obj = Behavior(trial.trajectories, trial.time_pad_off, trial.cm_per_pixel)
project.trialgroup.lineplot_traj(save_as=analysis_dir / f"{bodypart}_all_traj_with_beha_zones.svg", Beha=behavior_obj)


Lineplot all valid traj

In [ ]:
project.trialgroup.lineplot_traj(save_as=u.make_path(analysis_dir, f"{bodypart}_all_traj.svg"))

Lineplot all valid traj + facet grib per protocol

In [ ]:
project.trialgroup.lineplot_traj_per_indentity(save_as=analysis_dir / f"{bodypart}_traj_per_indentity.svg")

## Dstribution analysis (scalar metrics)

In [ ]:
metrics = [
    "avg_velocity_laser_period", 
    "tortuosity_laser_period", 
    "tortuosity_pad_to_lever", 
]

In [ ]:
comparisons = [
    # Conti vs Beta
    ("Conti_LaserOff.low",  "Beta_LaserOff.low"),
    ("Conti_LaserOff.high", "Beta_LaserOff.high"),
    ("Conti_LaserOn.low",   "Beta_LaserOn.low"),
    ("Conti_LaserOn.high",  "Beta_LaserOn.high"),

    # Off vs On
    ("Conti_LaserOff.low",  "Conti_LaserOn.low"),
    ("Conti_LaserOff.high", "Conti_LaserOn.high"),
    ("Beta_LaserOff.low",   "Beta_LaserOn.low"),
    ("Beta_LaserOff.high",  "Beta_LaserOn.high"),

    # low vs high
    ("Beta_LaserOff.low",   "Beta_LaserOff.high"),
    ("Beta_LaserOn.low",    "Beta_LaserOn.high"),
    ("Conti_LaserOff.low",  "Conti_LaserOff.high"),
    ("Conti_LaserOn.low",   "Conti_LaserOn.high"),
]

In [ ]:
for metric in metrics: 
    project.trialgroup.violin_statistic(comparisons, metric=metric, 
                                     save_as=u.make_path(analysis_dir / "distribution" / bodypart, f"{metric}.svg"))

Laser Off merged

In [ ]:
comparisons_merge_Off = [
    # conti vs beta
    ("Conti.low",  "Beta.low"),
    ("Conti.high", "Beta.high"),

    # low vs high
    ("LaserOff.high", "Conti.high"),
    ("LaserOff.high",  "Beta.high"),
    ("LaserOff.low",    "Beta.low"),
    ("LaserOff.low",   "Conti.low"),

    # condition high vs condition low
    ("Conti.low", "Conti.high"),
    ("Beta.low",  "Beta.high"),

    # Beta high ON vs Conti low ON 
    # (because conti low is suppose to be as damagefull as the beta high)
    ("Beta.high", "Conti.low"),
    ("Beta.low", "Conti.high")
]

In [ ]:
for metric in metrics: 
    project.trialgroup.violin_statistic(comparisons_merge_Off, metric=metric,  merge_Off=True,
                                     save_as=u.make_path(analysis_dir / "distribution" / bodypart, f"{metric}_merged_off.svg"),
                                     order=["LaserOff", "Conti", "Beta"])


## Single traj timeseries analysis

In [ ]:
for trial in tqdm(project.trialgroup.trials, desc="plotting timeseries"): 
    if not trial.is_valid(): 
        continue
    
    traj: TrajectoryDLC = trial.trajectories.get(bodypart)

    if not traj.is_valid(): 
        continue

    fig, ax = plt.subplots()

    ax = traj.plot_traj(traj.coords, ax, fig, color = "orange", label="raw")
    ax = traj.plot_traj(traj.interpolated_coords, ax, fig, color = "green", label="interpolated")
    ax = traj.plot_traj(traj.clean_coords, ax, fig, color = "blue", label="clean")


    name = trial.name
    title = name[:len(name)//2] + "\n" + name[len(name)//2:]

    ax.set(
        title=title,
        xlabel="X position (cm)",
        ylabel="Y position (cm)",
    )
    fig.savefig(u.make_path(analysis_dir / "traj_per_trial" / bodypart, f"{trial.name}.svg"))
    plt.close()

    for val in timeseries_metric: 

        traj.plot_tendency(value=val, time_pad_off=trial.time_pad_off,
                           laser_state=trial.laser_state,
                           group=trial.group,
                           show_angle= val == "angle",
                           save_as=u.make_path(analysis_dir / "tendency_per_trial" / bodypart / trial.name , f"{val}.svg"))
